# **Machine Learning dengan Scikit-Learn (CPMK M4)**

Materi Pokok:

1.	Preprocessing data: scaling, encoding, dan train-test split

2.	Model dasar: Linear Regression, Logistic Regression, dan K-Nearest Neighbor

3.	Evaluasi model: accuracy, precision, recall, dan mean squared error

4.	Cross-validation


Nama:Rinaldi Hamzah

NIM:5220411342

Program Studi:Informatika

In [1]:
#Mengakses Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Import Library**

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 4.1 Preprocessing data: scaling, encoding, dan train-test split

In [3]:
df = pd.read_excel('/content/drive/MyDrive/Semester7 2026/Data Science/pizza_sales_raw.xlsx', header=1)
#semua bentuk karakteristik data bisa di import seperti excel,csv,json
df.head() #melihat 5 baris data pertama

,pizza_id,order_id,pizza_name_id,quantity,order_date,order_time,unit_price,total_price,pizza_size,pizza_category,pizza_ingredients,pizza_name
0,1.0,1.0,hawaiian_m,1.0,1/1/2015,11:38:36,13.25,13.25,M,Classic,"Sliced Ham, Pineapple, Mozzarella Cheese",The Hawaiian Pizza
1,2.0,2.0,classic_dlx_m,1.0,1/1/2015,11:57:40,16.00,16.00,M,Classic,"Pepperoni, Mushrooms, Red Onions, Red Peppers,...",The Classic Deluxe Pizza
2,3.0,2.0,five_cheese_l,1.0,1/1/2015,11:57:40,18.50,18.50,L,Veggie,"Mozzarella Cheese, Provolone Cheese, Smoked Go...",The Five Cheese Pizza
3,4.0,2.0,ital_supr_l,1.0,1/1/2015,11:57:40,20.75,20.75,L,Supreme,"Calabrese Salami, Capocollo, Tomatoes, Red Oni...",The Italian Supreme Pizza
4,5.0,2.0,mexicana_m,1.0,1/1/2015,11:57:40,16.00,16.00,M,Veggie,"Tomatoes, Red Peppers, Jalapeno Peppers, Red O...",The Mexicana Pizza


Cleaning

In [4]:
#mengahpus angka nul
df['pizza_id'] = df['pizza_id'].astype(int)
df['order_id'] = df['order_id'].astype(int)
df['quantity'] = df['quantity'].astype(int)

In [5]:
df['unit_price'] = df['unit_price'].astype(int)
df['total_price'] = df['total_price'].astype(int)

In [6]:
df['order_date'] = pd.to_datetime(
    df['order_date'],
    format='mixed',
    dayfirst=True
)

In [7]:
df['order_time'] = pd.to_datetime(
    df['order_time'],
    format='mixed'
).dt.time

In [8]:
df['pizza_size'] = df['pizza_size'].str.lower().str.strip()
df['pizza_category'] = df['pizza_category'].str.lower().str.strip()
df['pizza_name'] = df['pizza_name'].str.lower().str.strip()
df['pizza_ingredients'] = (
    df['pizza_ingredients']
    .str.lower()
    .str.replace('[^a-z, ]', '', regex=True)
    .str.strip()
)

In [9]:
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48620 entries, 0 to 48619
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   pizza_id           48620 non-null  int64         
 1   order_id           48620 non-null  int64         
 2   pizza_name_id      48620 non-null  object        
 3   quantity           48620 non-null  int64         
 4   order_date         48620 non-null  datetime64[ns]
 5   order_time         48620 non-null  object        
 6   unit_price         48620 non-null  int64         
 7   total_price        48620 non-null  int64         
 8   pizza_size         48620 non-null  object        
 9   pizza_category     48620 non-null  object        
 10  pizza_ingredients  48620 non-null  object        
 11  pizza_name         48620 non-null  object        
dtypes: datetime64[ns](1), int64(5), object(6)
memory usage: 4.5+ MB


,pizza_id,order_id,pizza_name_id,quantity,order_date,order_time,unit_price,total_price,pizza_size,pizza_category,pizza_ingredients,pizza_name
0,1,1,hawaiian_m,1,2015-01-01,11:38:36,13,13,m,classic,"sliced ham, pineapple, mozzarella cheese",the hawaiian pizza
1,2,2,classic_dlx_m,1,2015-01-01,11:57:40,16,16,m,classic,"pepperoni, mushrooms, red onions, red peppers,...",the classic deluxe pizza
2,3,2,five_cheese_l,1,2015-01-01,11:57:40,18,18,l,veggie,"mozzarella cheese, provolone cheese, smoked go...",the five cheese pizza
3,4,2,ital_supr_l,1,2015-01-01,11:57:40,20,20,l,supreme,"calabrese salami, capocollo, tomatoes, red oni...",the italian supreme pizza
4,5,2,mexicana_m,1,2015-01-01,11:57:40,16,16,m,veggie,"tomatoes, red peppers, jalapeno peppers, red o...",the mexicana pizza


In [10]:
df.describe() #mendescribsikan bentuk data

,pizza_id,order_id,quantity,order_date,unit_price,total_price
count,48620.000000,48620.000000,48620.000000,48620,48620.000000,48620.000000
mean,24310.500000,10701.479761,1.019622,2015-06-29 11:03:43.611682560,16.037310,16.368161
min,1.000000,1.000000,1.000000,2015-01-01 00:00:00,9.000000,9.000000
25%,12155.750000,5337.000000,1.000000,2015-03-31 00:00:00,12.000000,12.000000
50%,24310.500000,10682.500000,1.000000,2015-06-28 00:00:00,16.000000,16.000000
75%,36465.250000,16100.000000,1.000000,2015-09-28 00:00:00,20.000000,20.000000
max,48620.000000,21350.000000,4.000000,2015-12-31 00:00:00,35.000000,83.000000
std,14035.529381,6180.119770,0.143077,NaN,3.514367,4.361458


In [11]:
df.isnull().sum() #menghitung jumlah data yang hilang

,0
pizza_id,0
order_id,0
pizza_name_id,0
quantity,0
order_date,0
order_time,0
unit_price,0
total_price,0
pizza_size,0
pizza_category,0


In [15]:
# Drop invalid dates
df = df.dropna(subset=['order_date'])

# Recalculate total_price (avoid data inconsistency)
df['total_price'] = df['quantity'] * df['unit_price']

In [16]:
# 4. FEATURE ENGINEERING
df['day_of_week'] = df['order_date'].dt.dayofweek
df['month'] = df['order_date'].dt.month
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

In [17]:
X = df[
    [
        'quantity',
        'unit_price',
        'pizza_size',
        'pizza_category',
        'day_of_week',
        'month',
        'is_weekend'
    ]
]

y = df['total_price']

In [18]:
# 6. PREPROCESSING PIPELINE
categorical_features = ['pizza_size', 'pizza_category']
numeric_features = [
    'quantity',
    'unit_price',
    'day_of_week',
    'month',
    'is_weekend'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', 'passthrough', numeric_features)
    ]
)

# 4.2 Model dasar: RandomForestRegressor

In [19]:
# 7. MODEL
model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ]
)

In [20]:
# 8. TRAIN-TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [21]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['pizza_size',
                                                   'pizza_category']),
                                                 ('num', 'passthrough',
                                                  ['quantity', 'unit_price',
                                                   'day_of_week', 'month',
                                                   'is_weekend'])])),
                ('model',
                 RandomForestRegressor(n_estimators=200, n_jobs=-1,
                                       random_state=42))])

In [22]:
y_pred = pipeline.predict(X_train)

In [23]:
y_pred = pipeline.predict(X_test)

In [24]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("=== MODEL EVALUATION ===")
print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R²   : {r2:.3f}")

=== MODEL EVALUATION ===
MAE  : 0.00
RMSE : 0.08
R²   : 1.000


In [25]:
feature_names = (
    pipeline.named_steps['preprocessor']
    .get_feature_names_out()
)

importances = pipeline.named_steps['model'].feature_importances_

feature_importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values(by='importance', ascending=False)

print("\n=== TOP 10 FEATURE IMPORTANCE ===")
print(feature_importance_df.head(10))


=== TOP 10 FEATURE IMPORTANCE ===
                        feature  importance
10              num__unit_price    0.692226
9                 num__quantity    0.293750
0             cat__pizza_size_l    0.011105
4           cat__pizza_size_xxl    0.001683
6   cat__pizza_category_classic    0.000271
3            cat__pizza_size_xl    0.000265
7   cat__pizza_category_supreme    0.000225
2             cat__pizza_size_s    0.000214
1             cat__pizza_size_m    0.000130
12                   num__month    0.000054


# 4.2 Evaluasi model: accuracy, precision, recall, dan mean squared error

In [26]:
median_sales = df['total_price'].median()
df['high_sales'] = (df['total_price'] >= median_sales).astype(int)

X = df[
    [
        'quantity',
        'unit_price',
        'pizza_size',
        'pizza_category',
        'day_of_week',
        'month',
        'is_weekend'
    ]
]

y = df['high_sales']

In [28]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ]
)

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

In [30]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')

print("=== CLASSIFICATION EVALUATION ===")
print(f"Accuracy  : {accuracy:.3f}")
print(f"Precision : {precision:.3f}")
print(f"Recall    : {recall:.3f}")

=== CLASSIFICATION EVALUATION ===
Accuracy  : 0.999
Precision : 0.998
Recall    : 0.999


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# 4.4 Cross-validation

In [32]:
from sklearn.model_selection import cross_val_score

In [35]:
cv_precision = cross_val_score(
    pipeline,
    X,
    y,
    cv=5,
    scoring='precision_weighted'
)

cv_recall = cross_val_score(
    pipeline,
    X,
    y,
    cv=5,
    scoring='recall_weighted'
)

print(f"Mean Precision : {cv_precision.mean():.3f}")
print(f"Mean Recall    : {cv_recall.mean():.3f}")

Mean Precision : 1.000
Mean Recall    : 1.000


# Tugas Praktikum:

2.	Bandingkan performa beberapa model menggunakan cross-validation.

**RANDOM FOREST**

In [36]:
# 7. RANDOM FOREST MODEL
# =========================================
rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', rf_model)
    ]
)

In [37]:
# 8. TRAIN-TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [38]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['pizza_size',
                                                   'pizza_category']),
                                                 ('num', 'passthrough',
                                                  ['quantity', 'unit_price',
                                                   'day_of_week', 'month',
                                                   'is_weekend'])])),
                ('model',
                 RandomForestRegressor(n_estimators=200, n_jobs=-1,
                                       random_state=42))])

In [39]:
y_pred = pipeline.predict(X_test)

In [40]:
# 11. EVALUATION
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("=== RANDOM FOREST REGRESSION EVALUATION ===")
print(f"MSE  : {mse:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"MAE  : {mae:.2f}")
print(f"R²   : {r2:.3f}")

=== RANDOM FOREST REGRESSION EVALUATION ===
MSE  : 0.00
RMSE : 0.00
MAE  : 0.00
R²   : 1.000


In [41]:
# 12. CROSS-VALIDATION
cv_scores = cross_val_score(
    pipeline,
    X,
    y,
    cv=5,
    scoring='neg_mean_squared_error'
)

cv_rmse = np.sqrt(-cv_scores)

print("\n=== CROSS-VALIDATION (5-FOLD) ===")
print(f"RMSE per fold : {cv_rmse}")
print(f"Mean RMSE    : {cv_rmse.mean():.2f}")
print(f"Std RMSE     : {cv_rmse.std():.2f}")


=== CROSS-VALIDATION (5-FOLD) ===
RMSE per fold : [0. 0. 0. 0. 0.]
Mean RMSE    : 0.00
Std RMSE     : 0.00


In [42]:
# 13. FEATURE IMPORTANCE
feature_names = (
    pipeline.named_steps['preprocessor']
    .get_feature_names_out()
)

importances = pipeline.named_steps['model'].feature_importances_

feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print("\n=== TOP 10 FEATURE IMPORTANCE ===")
print(feature_importance_df.head(10))


=== TOP 10 FEATURE IMPORTANCE ===
                        Feature  Importance
10              num__unit_price    0.972387
9                 num__quantity    0.027613
2             cat__pizza_size_s    0.000000
3            cat__pizza_size_xl    0.000000
0             cat__pizza_size_l    0.000000
1             cat__pizza_size_m    0.000000
5   cat__pizza_category_chicken    0.000000
4           cat__pizza_size_xxl    0.000000
7   cat__pizza_category_supreme    0.000000
6   cat__pizza_category_classic    0.000000
